In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 96 (delta 15), reused 23 (delta 4), pack-reused 56 (from 1)
Receiving objects: 100% (96/96), 43.03 MiB | 24.86 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [2]:
%cd /content/Recommendation-Systems-benchmarking/

/content/Recommendation-Systems-benchmarking


# Load data and candidate set

In [3]:
import pandas as pd
import numpy as np
import pickle

# Load train and test data
train= pd.read_csv("data/processed/train.csv")
test= pd.read_csv("data/processed/test.csv")

# Load movie metadata
movies=pd.read_csv("data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1")

# Load fixed candidate set
candidate_sets=pd.read_pickle("data/processed/candidate_sets_100.pkl")

print("Data loaded successfully")

Data loaded successfully


In [5]:
from llm_utils_py import (create_prompt,load_llm,generate_recommendations)

# Mistral-7B-Instruct

In [6]:
mistral_name = "mistralai/Mistral-7B-Instruct-v0.3"

mistral_tokenizer, mistral_model = load_llm(
    mistral_name
)

print("Mistral loaded")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral loaded


In [7]:
type(candidate_sets)

dict

In [8]:
candidate_sets[1].keys()

dict_keys(['test_item', 'items'])

In [9]:
user_id = 1

candidate_ids = candidate_sets[user_id]["items"]

candidate_movies = movies[
    movies["MovieID"].isin(candidate_ids)
]

In [10]:
print(type(candidate_movies))
print(candidate_movies.head())

<class 'pandas.core.frame.DataFrame'>
     MovieID                     Title                                Genres
47        48         Pocahontas (1995)  Animation|Children's|Musical|Romance
168      170            Hackers (1995)                 Action|Crime|Thriller
243      246        Hoop Dreams (1994)                           Documentary
260      263  Ladybird Ladybird (1994)                                 Drama
294      297            Panther (1995)                                 Drama


In [11]:
prompt = create_prompt(
    user_id=user_id,
    train=train,
    movies=movies,
    candidate_movies=candidate_movies
)

print(prompt)



You are a movie recommendation system.
Your task is to rank movies based on the user's previous preferences.

User's watched movies:
Back to the Future (1985) (Comedy|Sci-Fi) Rating: 5
Cinderella (1950) (Animation|Children's|Musical) Rating: 5
Christmas Story, A (1983) (Comedy|Drama) Rating: 5
Last Days of Disco, The (1998) (Drama) Rating: 5
Saving Private Ryan (1998) (Action|Drama|War) Rating: 5
Awakenings (1990) (Drama) Rating: 5
One Flew Over the Cuckoo's Nest (1975) (Drama) Rating: 5
Rain Man (1988) (Drama) Rating: 5
Sound of Music, The (1965) (Musical) Rating: 5
Ben-Hur (1959) (Action|Adventure|Drama) Rating: 5
Apollo 13 (1995) (Drama) Rating: 5
Mary Poppins (1964) (Children's|Comedy|Musical) Rating: 5
Dumbo (1941) (Animation|Children's|Musical) Rating: 5
Schindler's List (1993) (Drama|War) Rating: 5
Beauty and the Beast (1991) (Animation|Children's|Musical) Rating: 5
Bug's Life, A (1998) (Animation|Children's|Comedy) Rating: 5
Toy Story (1995) (Animation|Children's|Comedy) Rati

# Evaluation Loop

In [13]:
import os
import time
import pickle
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Path to save checkpoint
checkpoint_path = "/content/drive/MyDrive/mistral_checkpoint.pkl"

# -------------------------------------------------------
# Load checkpoint if it exists
# -------------------------------------------------------

if os.path.exists(checkpoint_path):

    with open(checkpoint_path, "rb") as f:
        mistral_results = pickle.load(f)

    print(f"Loaded checkpoint with {len(mistral_results)} users.")

else:

    mistral_results = {}
    print("No checkpoint found. Starting from scratch.")

# Evaluation loop
start_time=time.time()
total_users=len(candidate_sets)

for count,(user_id, data) in enumerate(candidate_sets.items(),start=1):

    # Skip already processed users
    if user_id in mistral_results:
        continue

    print(f"Processing user {count}/{total_users}")

    # Convert candidate MovieIDs to dataframe
    candidate_movies = movies[movies["MovieID"].isin(data["items"])]

    # Create prompt
    prompt=create_prompt(
        user_id=user_id,
        train=train,
        movies=movies,
        candidate_movies=candidate_movies)

    # Skip users with no history
    if prompt is None:
        continue

    # Generate recommendation
    response = generate_recommendations(
        prompt,
        mistral_tokenizer,
        mistral_model)

    # Store result
    mistral_results[user_id] = response

    # Save checkpoint every 50 users
    if len(mistral_results)%50== 0:
        with open(checkpoint_path, "wb") as f:
            pickle.dump(mistral_results, f)

        elapsed=(time.time()-start_time)/60

        print(f"Checkpoint saved ({len(mistral_results)} users)")
        print(f"Elapsed time: {elapsed:.2f} minutes")


# Save final results
with open(checkpoint_path, "wb") as f:
    pickle.dump(mistral_results, f)

print("----------------------------------")
print("Evaluation completed")
print(f"Total users processed: {len(mistral_results)}")
print("Results saved to:")
print(checkpoint_path)

Streaming output truncated to the last 5000 lines.
Processing user 1238/6040
Processing user 1239/6040
Processing user 1240/6040
Processing user 1241/6040
Processing user 1242/6040
Processing user 1243/6040
Processing user 1244/6040
Processing user 1245/6040
Processing user 1246/6040
Processing user 1247/6040
Processing user 1248/6040
Processing user 1249/6040
Processing user 1250/6040
Checkpoint saved (1250 users)
Elapsed time: 44.21 minutes
Processing user 1251/6040
Processing user 1252/6040
Processing user 1253/6040
Processing user 1254/6040
Processing user 1255/6040
Processing user 1256/6040
Processing user 1257/6040
Processing user 1258/6040
Processing user 1259/6040
Processing user 1260/6040
Processing user 1261/6040
Processing user 1262/6040
Processing user 1263/6040
Processing user 1264/6040
Processing user 1265/6040
Processing user 1266/6040
Processing user 1267/6040
Processing user 1268/6040
Processing user 1269/6040
Processing user 1270/6040
Processing user 1271/6040
Process

# Parse Mistral Output

In [14]:
import re

def parse_llm_ranking(response_text):
    """
    Extract MovieIDs from LLM response.

    Example output:
    1. MovieID: 123
    2. MovieID: 456

    Returns:
    [123,456]
    """

    movie_ids=re.findall(r"MovieID:\s*(\d+)",response_text)

    # Convert string IDs to integers
    movie_ids=[int(mid) for mid in movie_ids]

    return movie_ids

# Scoring Function

In [15]:
def llm_score_fn(user_id, items):

    # Get LLM response for this user
    response=mistral_results[user_id]

    # Parse ranking
    ranked_movies=parse_llm_ranking(response)

    # Assign scores
    movie_scores={}

    total_movies=len(ranked_movies)

    for position,movie_id in enumerate(ranked_movies):

        movie_scores[movie_id]=total_movies-position


    # Return scores in same order as candidate items
    scores=[]

    for movie_id in items:
        scores.append(movie_scores.get(movie_id, 0))

    return scores

# Metrics

In [16]:
# Hit@10=Checks whether the true test item appears in the top 10 recommended items.

def hit_at_k(ranked_items, test_item, k=10):
    if test_item in ranked_items[:k]:
        return 1.0

    return 0.0

# NDCG@10 = Measures ranking quality. Higher score is given when the true item appears higher in the top 10.

def ndcg_at_k(ranked_items, test_item, k=10):
    top_k=ranked_items[:k]

    # Check whether the true item is recommended
    if test_item in top_k:

        # Get position of true item (starting from 1)
        rank=top_k.index(test_item)+1

        # Discount score based on ranking position
        return 1.0/np.log2(rank+1)

    return 0.0

# Evaluate

In [17]:
# Evaluate a recommendation model.
    # This function will be used for all recommendation models.
    # It calculates the average Hit@10 and NDCG@10 using the fixed candidate set
def evaluate_model(score_fn, candidate_sets, k=10):
    hits=[]
    ndcgs=[]

    # Evaluate every user
    for user_id,candidate in candidate_sets.items():

        # Get 100 candidate movies
        items= candidate["items"]

        # Actual movie from test set
        test_item= candidate["test_item"]


        # Get model prediction scores
        scores= score_fn(user_id, items)

         # Rank movies according to predicted scores
        ranked_items= [item for item, score in sorted(
                zip(items, scores),
                key=lambda x: x[1],
                reverse=True)]


        # Calculate metrics
        hits.append(hit_at_k(ranked_items, test_item, k))

        ndcgs.append(
            ndcg_at_k(ranked_items, test_item, k)
        )
    # Return average performance over all users
    return {
        "Hit@10": np.mean(hits),
        "NDCG@10": np.mean(ndcgs)
    }

In [21]:
mistral_results_metrics=evaluate_model(llm_score_fn,candidate_sets,k=10)
print("Mistral Results")
print(f"Hit@10 : {mistral_results_metrics['Hit@10']:.4f}")
print(f"NDCG@10: {mistral_results_metrics['NDCG@10']:.4f}")

Mistral Results
Hit@10 : 0.0899
NDCG@10: 0.0438
